In [ ]:
from google.colab import files

# Summary

This project focuses on developing an AI-powered Industry-Specific Large Language Model (LLM) Bot for the Retail and E-commerce industry. The objective of the project was to create an intelligent customer complaint resolution system capable of understanding customer issues, classifying complaint categories, and identifying urgency levels using Natural Language Processing and Large Language Models.

The project was developed using a real-world e-commerce customer complaint dataset containing complaint descriptions, issue categories, and urgency labels. The dataset was cleaned, preprocessed, and converted into instruction-response format suitable for fine-tuning a pre-trained language model. The model selected for this project was TinyLlama, a lightweight and efficient Large Language Model available on Hugging Face

# Technologies Used

1. Python
2. Hugging Face Transformers
3. TinyLlama
4. LoRA (PEFT)








upload data

In [ ]:
uploaded = files.upload()

Saving train-00000-of-00001-20e0031a2e7c8400.parquet to train-00000-of-00001-20e0031a2e7c8400.parquet


In [ ]:
!pip install pandas pyarrow fastparquet transformers datasets accelerate peft trl bitsandbytes gradio scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.6 MB/s eta 0:00:00


Read Parquet Dataset

In [ ]:
import pandas as pd

df = pd.read_parquet('/content/drive/MyDrive/AlmaBetter/chatbot/train-00000-of-00001-20e0031a2e7c8400.parquet')

df.head()

,Complaint,Category,Urgency,target
0,I can’t add a new address — button does nothing.,Account & Shipping,Low,Account & Shipping | Low
1,Flash sale prices aren’t reflected during chec...,Promotions,Immediate,Promotions | Immediate
2,System declined my payment without any reason....,Payments,Immediate,Payments | Immediate
3,The shipping company marked it as delivered to...,Order Tracking,Medium,Order Tracking | Medium
4,Site logs me out randomly. which is unacceptable,Technical Issues,Medium,Technical Issues | Medium


Check Columns

In [ ]:
print(df.columns)

Index(['Complaint', 'Category', 'Urgency', 'target'], dtype='object')


Dataset Information

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1336 entries, 0 to 1335
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Complaint  1336 non-null   object
 1   Category   1336 non-null   object
 2   Urgency    1336 non-null   object
 3   target     1336 non-null   object
dtypes: object(4)
memory usage: 41.9+ KB


In [ ]:
df.shape

(1336, 4)

Clean Dataset

In [ ]:
df = df[['Complaint', 'Category', 'Urgency']]

df.dropna(inplace=True)

df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

print(df.shape)

(1336, 3)


In [ ]:
df.head()

,Complaint,Category,Urgency
0,I can’t add a new address — button does nothing.,Account & Shipping,Low
1,Flash sale prices aren’t reflected during chec...,Promotions,Immediate
2,System declined my payment without any reason....,Payments,Immediate
3,The shipping company marked it as delivered to...,Order Tracking,Medium
4,Site logs me out randomly. which is unacceptable,Technical Issues,Medium


Convert into Training Dataset

In [ ]:
training_data = []

for _, row in df.iterrows():

    instruction = f"""
Customer Complaint:
{row['Complaint']}

Identify the issue category and urgency level.
"""

    response = f"""
Category: {row['Category']}
Urgency: {row['Urgency']}
"""

    training_data.append({
        "instruction": instruction,
        "response": response
    })

print(training_data[0])

{'instruction': '\nCustomer Complaint:\nI can’t add a new address — button does nothing.\n\nIdentify the issue category and urgency level.\n', 'response': '\nCategory: Account & Shipping\nUrgency: Low\n'}


Convert to Hugging Face Dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(training_data)

dataset

Dataset({
    features: ['instruction', 'response'],
    num_rows: 1336
})

Install Additional Libraries

In [ ]:
!pip install -U transformers accelerate peft trl bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 30.1 MB/s eta 0:00:00


Import Libraries

In [ ]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model
)

from trl import SFTTrainer

Load TinyLlama Model

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)

model = model.to("cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Format Dataset Properly

In [ ]:
def formatting_func(example):

    text = f"""
### Instruction:
{example['instruction']}

### Response:
{example['response']}
"""

    return text

Configure LoRA

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

Training Arguments

In [ ]:
training_arguments = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    num_train_epochs=3,

    logging_steps=10,

    fp16=True,

    save_strategy="epoch",

    report_to="none"
)

Create Trainer

In [ ]:
!pip uninstall -y torchao
!pip install torchao>=0.16.0 -q

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    formatting_func=formatting_func,
    args=training_arguments
)

Applying formatting function to train dataset:   0%|          | 0/1336 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1336 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1336 [00:00<?, ? examples/s]

START TRAINING

In [ ]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,2.680599
20,1.426694
30,0.733872
40,0.717596
50,0.609780
60,0.624852
70,0.627095
80,0.607438
90,0.636187
100,0.605578


TrainOutput(global_step=1002, training_loss=0.4920766072834799, metrics={'train_runtime': 570.7393, 'train_samples_per_second': 7.022, 'train_steps_per_second': 1.756, 'total_flos': 1512667359350784.0, 'train_loss': 0.4920766072834799})

# Save Model

In [ ]:
trainer.model.save_pretrained("ecommerce-llm-model")

tokenizer.save_pretrained("ecommerce-llm-model")

('ecommerce-llm-model/tokenizer_config.json',
 'ecommerce-llm-model/chat_template.jinja',
 'ecommerce-llm-model/tokenizer.json')

In [ ]:
import os

os.listdir("ecommerce-llm-model")

['README.md',
 'tokenizer.json',
 'adapter_model.safetensors',
 'chat_template.jinja',
 'adapter_config.json',
 'tokenizer_config.json']

# Test Model

In [ ]:
prompt = """
Customer Complaint:
Transaction failed but my card got debited anyway.

Identify the issue category and urgency level.
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

[transformers] Both `max_new_tokens` (=50) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Customer Complaint:
Transaction failed but my card got debited anyway.

Identify the issue category and urgency level.

Category: Payments
Urgency: High




# Create Chatbot Function

In [ ]:
!pip install huggingface_hub -q

In [ ]:
from huggingface_hub import logout, login

logout()

login()

In [ ]:
from huggingface_hub import whoami

whoami()

{'type': 'user',
 'id': '6a042a0a4a6414e82b52ba23',
 'name': 'Kashfur',
 'fullname': 'Kashfur Rahman Khan',
 'email': 'kashfurk@gmail.com',
 'emailVerified': True,
 'canPay': False,
 'billingMode': 'prepaid',
 'periodEnd': 1780272000,
 'isPro': False,
 'avatarUrl': '/avatars/c644d99812b73120abeb17cacc793416.svg',
 'orgs': [],
 'auth': {'type': 'access_token',
  'accessToken': {'displayName': 'kashfur',
   'role': 'write',
   'createdAt': '2026-05-13T07:46:31.864Z'}}}

In [ ]:
trainer.model.push_to_hub("Kashfur/ecommerce-llm-bot")

tokenizer.push_to_hub("Kashfur/ecommerce-llm-bot")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   6%|6         |  563kB / 9.02MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Kashfur/ecommerce-llm-bot/commit/1a263f165ae399a9c2bd58ca5a96288fe9d53d3c', commit_message='Upload tokenizer', commit_description='', oid='1a263f165ae399a9c2bd58ca5a96288fe9d53d3c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kashfur/ecommerce-llm-bot', endpoint='https://huggingface.co', repo_type='model', repo_id='Kashfur/ecommerce-llm-bot'), pr_revision=None, pr_num=None)

In [ ]:
from huggingface_hub import upload_folder

upload_folder(
    folder_path="ecommerce-llm-model",
    repo_id="Kashfur/ecommerce-llm-bot",
    repo_type="model"
)

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors: 100%|##########| 9.02MB / 9.02MB            

CommitInfo(commit_url='https://huggingface.co/Kashfur/ecommerce-llm-bot/commit/9143e0d74d0ec9301cca7ee6b4d704652388c2d5', commit_message='Upload folder using huggingface_hub', commit_description='', oid='9143e0d74d0ec9301cca7ee6b4d704652388c2d5', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Kashfur/ecommerce-llm-bot', endpoint='https://huggingface.co', repo_type='model', repo_id='Kashfur/ecommerce-llm-bot'), pr_revision=None, pr_num=None)

ChatBot Link - https://huggingface.co/spaces/Kashfur/ecommerce-llm-chatbot

# Project Outcome

The final outcome of the project is a publicly accessible AI-powered e-commerce complaint resolution chatbot capable of analyzing customer complaints and generating contextually relevant predictions. This project demonstrates the practical implementation of Large Language Models for real-world business applications in the Retail and E-commerce domain.